# 02dashdata: small summary files for the dashboard

Reads the clean tables in `data/clean` and writes small summary files to `data/app`, which is all the Streamlit app reads.
Keeping the app's data small makes it start fast (also on Streamlit Community Cloud).

| Output | What one row is |
|---|---|
| `daily_network.parquet` | one day: network boardings and alightings, day type and holiday name |
| `monthly_route.parquet` | one month and route: average daily boardings (all days, weekdays, weekends) |
| `daily_line.parquet` | one day and line: boardings and alightings (for the route and operator filters) |
| `monthly_station.parquet` | one month, line and station: average daily boardings (all days, weekdays) and days with data |
| `monthly_network.parquet` | one month: averages for all days, weekdays and weekends, and the number of stations |
| `crowding_profile.parquet` | one quarter, day type and half hour: network average and highest crowding |
| `hourly_line.parquet` | one line, day type, gate (board or alight) and hour: average passengers per day (Seoul Metro, 2024) |
| `hourly_station.parquet` | one station, day type, gate and hour: average passengers per day (Seoul Metro, 2024) |
| `crowding_station.parquet` | one quarter, day type, line, station, direction and half hour: crowding % (for the heatmap) |
| `crowding_peaks.parquet` | one quarter and day type: the ten most crowded station, direction and time combinations |
| `lines.csv`, `stations.csv`, `operations.csv` | copied from `data/clean` |

**Day types.** A *weekday* is Monday to Friday that is not a public holiday. Public holidays come from the `holidays` package (Korean calendar, including lunar holidays and substitute days), plus any extra dates listed in Section 0.

## Section 0: settings

In [1]:
from pathlib import Path

CLEAN = Path("data/clean")                   # <- clean tables from 01clean
APP = Path("data/app")                       # <- output folder for the dashboard

# Extra non-working days not in the holidays package (date: name). Found by the check in Section 2.
# Workers' Day (1 May) closed many workplaces before it became a public holiday in 2026.
EXTRA_HOLIDAYS = {                           # <- add or remove dates here
    "2015-05-01": "Workers' Day",
    "2017-05-01": "Workers' Day",
    "2018-05-01": "Workers' Day",
    "2019-05-01": "Workers' Day",
    "2020-05-01": "Workers' Day",
    "2023-05-01": "Workers' Day",
    "2024-05-01": "Workers' Day",
    "2025-05-01": "Workers' Day",
    "2025-01-31": "Bridge day after Seollal",
    "2025-10-10": "Bridge day after Chuseok and Hangeul Day",
}
HOLIDAY_LANGUAGE = "en_US"                   # <- language of holiday names ("ko" for Korean)

# Check in Section 2: flag weekdays whose ridership is below this share of nearby weekdays
LOW_DAY_THRESHOLD = 0.85                     # <-

N_PEAKS = 10                                 # <- most crowded combinations kept per quarter and day type

# Hourly data: months left out because of errors in the source file (month: reason)
HOURLY_EXCLUDE = {"2024-04": "13:00 slot about ten times too high on every day of the month"}   # <-


## Section 1: load

In [2]:
import shutil
import holidays
import pandas as pd

APP.mkdir(parents=True, exist_ok=True)
rid = pd.read_parquet(CLEAN / "ridership_daily.parquet")
crowd = pd.read_parquet(CLEAN / "crowding.parquet")
lines = pd.read_csv(CLEAN / "lines.csv")
stations = pd.read_csv(CLEAN / "stations.csv")
for df in (rid, crowd):
    for c in df.columns:
        if str(df[c].dtype) == "category":
            df[c] = df[c].astype(str)
print(f"ridership {len(rid):,} rows, crowding {len(crowd):,} rows, {len(lines)} lines, {len(stations)} stations")

ridership 2,536,679 rows, crowding 650,637 rows, 27 lines, 633 stations


## Section 2: day types and holiday check

Every date gets a day type: `weekday`, `weekend` or `holiday`. Then a check: weekdays (not holidays) with unusually low ridership
compared with the weekdays around them. These are often missing holidays (for example a one-off temporary holiday) or special events.

In [3]:
years = range(rid["date"].dt.year.min(), rid["date"].dt.year.max() + 1)
kr = holidays.KR(years=years, language=HOLIDAY_LANGUAGE)
extra = {pd.Timestamp(k): v for k, v in EXTRA_HOLIDAYS.items()}

daily = rid.groupby("date", as_index=False)[["board", "alight"]].sum()
daily["holiday_name"] = daily["date"].map(lambda d: extra.get(d, kr.get(d)))
daily["day_type"] = "weekday"
daily.loc[daily["date"].dt.dayofweek >= 5, "day_type"] = "weekend"
daily.loc[daily["holiday_name"].notna(), "day_type"] = "holiday"
print(daily["day_type"].value_counts().to_string())

wk = daily[daily["day_type"] == "weekday"].set_index("date")["board"]
ratio = wk / wk.rolling(21, center=True, min_periods=5).median()
low = ratio[ratio < LOW_DAY_THRESHOLD].round(2)
print(f"weekdays below {LOW_DAY_THRESHOLD:.0%} of nearby weekdays: {len(low)} (2023-08-10 is Typhoon Khanun, a real disruption, so it stays a weekday)")
low.to_frame("share_of_nearby_weekdays")

day_type
weekday    2891
weekend    1174
holiday     226
weekdays below 85% of nearby weekdays: 1 (2023-08-10 is Typhoon Khanun, a real disruption, so it stays a weekday)


C:\Users\likre\AppData\Local\Temp\ipykernel_53292\4133385788.py:8: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  daily.loc[daily["date"].dt.dayofweek >= 5, "day_type"] = "weekend"


,share_of_nearby_weekdays
date,
2023-08-10,0.81


## Section 3: monthly network, route and station summaries

In [4]:
rid = rid.merge(daily[["date", "day_type"]], on="date")
rid["month"] = rid["date"].dt.to_period("M").dt.to_timestamp()
daily["month"] = daily["date"].dt.to_period("M").dt.to_timestamp()

# network: average daily boardings per month by day type
net = daily.pivot_table(index="month", columns="day_type", values="board", aggfunc="mean")
net = net.rename(columns={"weekday": "avg_weekday", "weekend": "avg_weekend", "holiday": "avg_holiday"})
net["avg_all_days"] = daily.groupby("month")["board"].mean()
net["days"] = daily.groupby("month")["date"].nunique()
net["weekdays"] = daily[daily["day_type"] == "weekday"].groupby("month")["date"].nunique()
net["stations"] = rid.groupby("month").apply(lambda g: len(g[["line_ko", "station_ko"]].drop_duplicates()), include_groups=False)
net = net.reset_index()
net.columns.name = None
print(f"monthly network: {len(net)} months")

# route: average daily boardings per month
rid = rid.merge(lines[["line_ko", "route"]], on="line_ko", how="left")
route_day = rid.groupby(["date", "month", "day_type", "route"], as_index=False)["board"].sum()
route = route_day.pivot_table(index=["month", "route"], columns="day_type", values="board", aggfunc="mean").reset_index()
route = route.rename(columns={"weekday": "avg_weekday", "weekend": "avg_weekend", "holiday": "avg_holiday"})
route["avg_all_days"] = route_day.groupby(["month", "route"])["board"].mean().values
route.columns.name = None
print(f"monthly route: {len(route):,} rows")

# station: average daily boardings per month (all days and weekdays) and days with data
st_all = rid.groupby(["month", "line_ko", "station_ko"]).agg(avg_all_days=("board", "mean"), days=("date", "nunique"))
st_wk = rid[rid["day_type"] == "weekday"].groupby(["month", "line_ko", "station_ko"]).agg(avg_weekday=("board", "mean"), weekdays=("date", "nunique"))
station = st_all.join(st_wk).reset_index()
station = station.merge(stations[["line_ko", "station_ko", "station_en", "station_label"]], on=["line_ko", "station_ko"], how="left")
print(f"monthly station: {len(station):,} rows, missing labels: {station['station_label'].isna().sum()}")

# line: daily boardings per line (the app adds route and operator from lines.csv)
daily_line = rid.groupby(["date", "line_ko"], as_index=False)[["board", "alight"]].sum()
print(f"daily line: {len(daily_line):,} rows")


monthly network: 141 months
monthly route: 2,155 rows
monthly station: 84,137 rows, missing labels: 0
daily line: 108,357 rows


## Section 4: crowding summaries

Uses Seoul Metro stations only (branch-point entries left out). The network profile is the average over all stations and directions in each half hour.

In [5]:
cw = crowd[~crowd["branch_entry"]].dropna(subset=["crowding_pct"])
profile = cw.groupby(["quarter", "day_type", "slot"], as_index=False).agg(avg_pct=("crowding_pct", "mean"), max_pct=("crowding_pct", "max"))
slot_order = {s: i for i, s in enumerate(sorted(cw["slot"].unique(), key=lambda s: (s < "05", s)))}
profile["slot_order"] = profile["slot"].map(slot_order)
profile = profile.sort_values(["quarter", "day_type", "slot_order"], ignore_index=True)

peaks = (cw.sort_values("crowding_pct", ascending=False).groupby(["quarter", "day_type"]).head(N_PEAKS))
peaks = peaks.merge(stations[["line_ko", "station_ko", "station_label"]], on=["line_ko", "station_ko"], how="left")
peaks = peaks.sort_values(["quarter", "day_type", "crowding_pct"], ascending=[True, True, False], ignore_index=True)
crowd_station = crowd[~crowd["branch_entry"]].merge(stations[["line_ko", "station_ko", "station_label"]], on=["line_ko", "station_ko"], how="left")
crowd_station = crowd_station[["quarter", "day_type", "line_ko", "station_code", "station_label", "direction", "slot", "crowding_pct"]]
print(f"crowding profile: {len(profile):,} rows, peaks: {len(peaks)} rows, station rows: {len(crowd_station):,}, missing labels: {crowd_station['station_label'].isna().sum()}")
peaks[(peaks["day_type"] == "weekday")].groupby("quarter").head(1)[["quarter", "line_ko", "station_label", "direction", "slot", "crowding_pct"]].round(1)

crowding profile: 1,162 rows, peaks: 300 rows, station rows: 642,213, missing labels: 0


,quarter,line_ko,station_label,direction,slot,crowding_pct
20,2023Q4,4호선,Hansung Univ. (한성대입구),down,08:00,166.2
50,2024Q1,7호선,Gasan Digital Complex (가산디지털단지),down,18:00,157.0
80,2024Q2,2호선,Sadang (사당),outer,08:30,148.3
110,2024Q4,2호선,Sadang (사당),outer,09:00,154.8
140,2025Q1,8호선,Mongchontoseong (몽촌토성),down,08:30,151.9
170,2025Q2,2호선,Seocho (서초),inner,18:00,145.7
200,2025Q3,2호선,Sadang (사당),outer,08:30,151.5
230,2025Q4,8호선,Gangdonggu Office (강동구청),down,08:00,159.4
260,2026Q1,2호선,Sadang (사당),outer,08:30,147.7
290,2026Q2,2호선,Sadang (사당),outer,08:30,144.6


## Section 5: hourly summaries (Seoul Metro Lines 1 to 8, 2024)

Average passengers per day for each hour, by day type and gate (board or alight). `hour` is the start of the hour; 5 means before 06:00 and 24 means after midnight.

April 2024 is left out (see `HOURLY_EXCLUDE` in Section 0): its 13:00 slot is about ten times too high in the source file. Without April, the hourly totals match the daily card data to within 0.5%.

In [6]:
hr = pd.read_parquet(CLEAN / "hourly.parquet")
for c in hr.columns:
    if str(hr[c].dtype) == "category":
        hr[c] = hr[c].astype(str)
bad = hr["date"].dt.strftime("%Y-%m").isin(list(HOURLY_EXCLUDE))
print(f"left out {bad.sum():,} rows from {sorted(HOURLY_EXCLUDE)}: {list(HOURLY_EXCLUDE.values())}")
hr = hr[~bad]
hr = hr.merge(daily[["date", "day_type"]], on="date", how="left")
n_days = hr.groupby("day_type")["date"].nunique()
print(f"hourly: {len(hr):,} rows, days per type: {n_days.to_dict()}")

hourly_line = hr.groupby(["line_ko", "day_type", "gate", "hour"], as_index=False)["passengers"].sum()
hourly_line["avg_per_day"] = hourly_line["passengers"] / hourly_line["day_type"].map(n_days)
hourly_line = hourly_line.drop(columns="passengers")

hourly_station = hr.groupby(["line_ko", "station_code", "station_ko", "day_type", "gate", "hour"], as_index=False)["passengers"].mean()
hourly_station = hourly_station.rename(columns={"passengers": "avg_per_day"})
hourly_station = hourly_station.merge(stations[["line_ko", "station_ko", "station_label"]], on=["line_ko", "station_ko"], how="left")
print(f"hourly line: {len(hourly_line):,} rows, hourly station: {len(hourly_station):,} rows, missing labels: {hourly_station['station_label'].isna().sum()}")

left out 326,440 rows from ['2024-04']: ['13:00 slot about ten times too high on every day of the month']
hourly: 3,662,040 rows, days per type: {'holiday': 19, 'weekday': 224, 'weekend': 93}
hourly line: 960 rows, hourly station: 32,840 rows, missing labels: 0


## Section 6: save

In [7]:
daily.to_parquet(APP / "daily_network.parquet", index=False)
net.to_parquet(APP / "monthly_network.parquet", index=False)
daily_line.to_parquet(APP / "daily_line.parquet", index=False)
route.to_parquet(APP / "monthly_route.parquet", index=False)
station.to_parquet(APP / "monthly_station.parquet", index=False)
profile.to_parquet(APP / "crowding_profile.parquet", index=False)
peaks.to_parquet(APP / "crowding_peaks.parquet", index=False)
cs = crowd_station.copy()
for c in ["quarter", "day_type", "line_ko", "station_label", "direction", "slot"]:
    cs[c] = cs[c].astype("category")
cs.to_parquet(APP / "crowding_station.parquet", index=False)
hourly_line.to_parquet(APP / "hourly_line.parquet", index=False)
hourly_station.to_parquet(APP / "hourly_station.parquet", index=False)
shutil.copy(CLEAN / "lines.csv", APP / "lines.csv")
shutil.copy(CLEAN / "stations.csv", APP / "stations.csv")
shutil.copy(CLEAN / "operations.csv", APP / "operations.csv")
total = 0
for f in sorted(APP.iterdir()):
    total += f.stat().st_size
    print(f"  {f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")
print(f"  {'total':28s} {total / 1e6:6.2f} MB")

  crowding_peaks.parquet         0.01 MB
  crowding_profile.parquet       0.02 MB
  crowding_station.parquet       1.70 MB
  daily_line.parquet             1.42 MB
  daily_network.parquet          0.10 MB
  hourly_line.parquet            0.01 MB
  hourly_station.parquet         0.23 MB
  lines.csv                      0.00 MB
  monthly_network.parquet        0.01 MB
  monthly_route.parquet          0.07 MB
  monthly_station.parquet        1.36 MB
  operations.csv                 0.00 MB
  stations.csv                   0.06 MB
  total                          5.00 MB


___
**END**